# Σημειώσεις Pandas: Σύνδεση Πίνακα με τον Εαυτό του (`Self-Merge`)

Το **Self-Merge** είναι η ένωση ενός DataFrame με τον εαυτό του (`pd.merge(df, df, ...)`). Χρησιμοποιείται για **ιεραρχικές σχέσεις** (π.χ. υπάλληλος–προϊστάμενος) ή για **σύγκριση γραμμών μεταξύ τους** μέσα στην ίδια ομάδα.

⚠️ **Κανόνας:** Επειδή και οι δύο πλευρές έχουν ακριβώς τα ίδια ονόματα στηλών, η παράμετρος **`suffixes=`** είναι απαραίτητη για να μη χαθείς ανάμεσα στα `_x` και `_y`.

---

## 💻 Περίπτωση 1: Ιεραρχία (Υπάλληλος – Manager)

Ο πίνακας `employees` περιέχει και τους υπαλλήλους και τους managers. Χρησιμοποιούμε `how="left"` ώστε να μην εξαφανιστεί ο CEO (που δεν έχει manager).

```python
import pandas as pd

emp_with_managers = pd.merge(
    employees,                       # Αριστερά: Ο Υπάλληλος
    employees,                       # Δεξιά: Ο Manager
    how="left",
    left_on="manager_id",            # Το ID του manager του υπαλλήλου
    right_on="id",                   # Το προσωπικό ID του manager
    suffixes=("_emp", "_mgr")        # Διαχωρισμός στηλών
)

# Επιλογή των στηλών που μας ενδιαφέρουν
emp_with_managers[["name_emp", "name_mgr"]]
```

---

## 💻 Περίπτωση 2: Σύγκριση Ζευγαριών στην Ίδια Ομάδα

**Ερώτηση:** *«Βρες ποιοι υπάλληλοι κερδίζουν περισσότερα από κάποιον συνάδελφό τους στο ίδιο τμήμα.»* (Μοτίβο: **Merge ➡️ Pair ➡️ Filter**).

```python
# 1. Merge & Pair: Φτιάχνουμε όλα τα ζευγάρια ανά τμήμα
pairs = pd.merge(
    employees, 
    employees, 
    on="department", 
    suffixes=("_1", "_2")
)

# 2. Filter: Κρατάμε μόνο όσους έχουν μεγαλύτερο μισθό από τον συνάδελφό τους
higher_earners = pairs[pairs["salary_1"] > pairs["salary_2"]]
```


In [12]:
import pandas as pd
import numpy as np

employees = pd.DataFrame({
    'id':[1, 2, 3, 4, 5, 6],
    'name': ['Eleni (CEO)', 'Nikos (Tech Mgr)', 'Maria (Sales Mgr)', 'Giorgos', 'Anna', 'Kostas'],
    'department': ['Management', 'Tech', 'Sales', 'Tech', 'Tech', 'Sales'],
    'salary':[30000, 25000, 22000, 14000, 15000, 17000],
    'manager_id': [np.nan, 1, 1, 2, 2, 3]
})

employees


,id,name,department,salary,manager_id
0,1,Eleni (CEO),Management,30000,NaN
1,2,Nikos (Tech Mgr),Tech,25000,1.0
2,3,Maria (Sales Mgr),Sales,22000,1.0
3,4,Giorgos,Tech,14000,2.0
4,5,Anna,Tech,15000,2.0
5,6,Kostas,Sales,17000,3.0


In [13]:
self_join = pd.merge(employees, employees, how='left', left_on='manager_id', right_on='id', suffixes=('_emp','_mgr'))


In [14]:
self_join[['name_emp', 'salary_emp', 'name_mgr', 'salary_mgr']]

,name_emp,salary_emp,name_mgr,salary_mgr
0,Eleni (CEO),30000,NaN,NaN
1,Nikos (Tech Mgr),25000,Eleni (CEO),30000.0
2,Maria (Sales Mgr),22000,Eleni (CEO),30000.0
3,Giorgos,14000,Nikos (Tech Mgr),25000.0
4,Anna,15000,Nikos (Tech Mgr),25000.0
5,Kostas,17000,Maria (Sales Mgr),22000.0


In [ ]:
#στην ουσία χρησιμοποιώ την λογική των cte

df = employees.groupby('department')['salary'].min().reset_index(name='min')
merged = pd.merge(employees, df, how='inner', on='department')

merged.loc[merged['salary']>merged['min'], ['name', 'salary', 'min']]

,name,salary,min
1,Nikos (Tech Mgr),25000,14000
2,Maria (Sales Mgr),22000,17000
4,Anna,15000,14000
